In [1]:
import numpy as np

from utils import load_preprocessed_ds

In [2]:
X_tr, y_tr, X_te, y_te, X_tr_pd = load_preprocessed_ds()

In [3]:
rng = np.random.default_rng(44)

In [4]:
# Boostrap sampling function
def bootstrap_sample(X, y, rng):
    n_samples = X.shape[0]
    sampled_indices = rng.choice(n_samples, size=n_samples, replace=True)

    X_boot = X[sampled_indices]
    y_boot = y[sampled_indices]

    bootstrap_indices = np.unique(sampled_indices)
    all_indices = np.arange(n_samples)
    sample_mask = np.isin(all_indices, bootstrap_indices)
    oob_indices = all_indices[~sample_mask]

    # print(len(oob_indices)/n_samples) # Should be ~ 0.37
    # print(np.intersect1d(bootstrap_indices, oob_indices)) # Should be empty
    return X_boot, y_boot, oob_indices

bootstrap_sample(X_tr, y_tr, rng)

(array([[ 1.69149923,  1.        ,  1.76494012, ...,  1.        ,
          0.        ,  0.        ],
        [ 2.34470585,  0.        ,  0.57404228, ...,  0.        ,
          0.        ,  0.        ],
        [-1.46566609,  1.        , -1.09321469, ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 1.03829261,  1.        , -0.61685555, ...,  1.        ,
          0.        ,  0.        ],
        [-0.26812062,  1.        , -0.14049642, ...,  1.        ,
          0.        ,  0.        ],
        [ 0.7116893 ,  0.        ,  0.87176674, ...,  0.        ,
          0.        ,  0.        ]], shape=(242, 20)),
 array([0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 1., 0., 0., 1.,
        1., 0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
        1., 0., 1., 0., 1., 1., 1., 0., 1., 0., 0., 1., 1., 1., 1., 1., 0.,
        0., 1., 1., 1., 0., 0., 0., 1., 0., 0., 1., 1., 1., 0., 1., 1., 0.,
        1., 1., 1., 0., 1., 1., 1., 1., 0., 0., 1., 

In [12]:
MIN_SAMPLES_SPLIT = 10
MAX_DEPTH = 3
MAX_FEATURES = 3

In [40]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.is_leaf = self.value is not None

    def predict_one(self, x):
        if self.is_leaf:
            return self.value
        else:
            if x[self.feature] <= self.threshold:
                return self.left.predict_one(x)
            else:
                return self.right.predict_one(x)

    def predict_batch(self, X):
        return np.array([self.predict_one(X[i,:]) for i in range(X.shape[0])])


def most_common_label(y):
    vals, counts = np.unique(y, return_counts=True)
    return vals[np.argmax(counts)]

def split_labels(y, feature, threshold):
    y_left = y[feature <= threshold]
    y_right = y[feature > threshold]
    return y_left, y_right

def gini(y): # y is labels
    labels_count = len(y)
    if labels_count == 0:
        return 0
    _, frequencies = np.unique(y, return_counts=True)
    probabilities = np.sum((frequencies / labels_count) ** 2)
    return 1 - probabilities

def weighted_gini(y_left, y_right):
    len_left = len(y_left)
    len_right = len(y_right)
    len_total = len_left + len_right
    return (len_left * gini(y_left) + len_right * gini(y_right)) / len_total

def find_best_split(X, y, rng, max_features=MAX_FEATURES):
    lowest_gini = np.inf
    feature_at_lowest_gini = None
    thresh_at_lowest_gini = None

    n_features = X.shape[1]
    if max_features > n_features:
        raise Exception("max_features cannot be bigger than n_features!")

    random_features = rng.choice(n_features, size=max_features, replace=False)
    print(random_features)
    for feature_idx in random_features:

        unique_sorted_vals = np.unique(X[:, feature_idx])
        thresholds = (unique_sorted_vals[:-1] + unique_sorted_vals[1:]) / 2

        for threshold in thresholds:
            y_left, y_right = split_labels(y, X[:,feature_idx], threshold)
            current_gini = weighted_gini(y_left, y_right)

            if current_gini < lowest_gini:
                lowest_gini = current_gini
                feature_at_lowest_gini = feature_idx
                thresh_at_lowest_gini = threshold
    return lowest_gini, feature_at_lowest_gini, thresh_at_lowest_gini

find_best_split(X_tr, y_tr, rng)

def build_tree(X, y, rng, max_features=MAX_FEATURES, depth=0, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES_SPLIT):

    # stopping conditions
    reached_max_depth = depth == max_depth
    too_little_samples_left = len(y) < min_samples_split
    all_samples_same_class = len(np.unique(y)) == 1
    if reached_max_depth or too_little_samples_left or all_samples_same_class:
        return Node(value=most_common_label(y)) # leaf

    lowest_weighted_gini, feature, thresh = find_best_split(X, y, rng, max_features)

    # another stopping condition
    no_improvement_in_splitting = lowest_weighted_gini >= gini(y)
    if no_improvement_in_splitting:
        return Node(value=most_common_label(y)) # leaf

    # split X and y
    mask = X[:, feature] <= thresh
    X_left = X[mask, :]
    y_left = y[mask]
    X_right = X[~mask, :]
    y_right = y[~mask]

    # recurse
    left_node = build_tree(X_left, y_left, rng, max_features, depth + 1, max_depth, min_samples_split)
    right_node = build_tree(X_right, y_right, rng, max_features, depth + 1, max_depth, min_samples_split)

    return Node(feature=feature, threshold=thresh, left=left_node, right=right_node)

[10 14  0]


In [68]:
root = build_tree(X_tr, y_tr, rng, 3)
preds = root.predict_batch(X_te)
accuracy = np.mean(preds == y_te)
print(f"accuracy: {accuracy}")

[11 14  9]
[ 9  8 12]
[ 8 10 12]
[6 2 8]
[ 0 11  7]
[0 1 5]
[ 1 11 15]
accuracy: 0.7377049180327869
